# Spot2 — Final Lead Opportunity Assessment

**Post-recovery clean-room notebook**

This notebook is intentionally source-of-truth driven. It reads only frozen artifacts under `AssessmentSol1/**`.

Final system:
- scoring moment: **T1 / first inquiry**
- target: `T1_FIRST_INQUIRY_EVENTUAL_SCHEDULED_VISIT_V1`
- Lead Quality: `LQ_RECOVERY_R4_STATIC_MATCH_V1`
- Opportunity Score: `OPPORTUNITY_ACTIONABILITY_GATE_V2_FROZEN_2026-08-30`
- capacity: **P80 / top 20%**
- fallback: **K=3**

Historical E018/E019/E020 evidence is supporting only unless reproduced inside AssessmentSol1.

In [ ]:
from pathlib import Path
import csv, json

def find_repo_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for p in candidates:
        if (p / "AssessmentSol1").exists():
            return p
    raise RuntimeError("Could not locate repository root containing AssessmentSol1/")

ROOT = find_repo_root()
A = ROOT / "AssessmentSol1"

snapshot = json.loads((A / "final" / "source_snapshot.json").read_text())
post = json.loads((A / "recovery_downstream" / "POST_RECOVERY_FINAL_STATE.json").read_text())
score_cfg = json.loads((A / "opportunity_score" / "frozen_score_config.json").read_text())
audit = json.loads((A / "audit" / "final_audit.json").read_text())
llm_gate = json.loads((A / "llm" / "results" / "prompt12_gate.json").read_text())

print("Source snapshot:", snapshot["snapshot_version"])
print("Final audit:", audit["status"], "blockers:", audit["gate"]["blocker_count"])

## 1. Pre-packaging consistency gate

The notebook must stop if any frozen authority differs from the post-recovery system.

In [ ]:
EXPECTED = {
    "lead_quality": "LQ_RECOVERY_R4_STATIC_MATCH_V1",
    "opportunity": "OPPORTUNITY_ACTIONABILITY_GATE_V2_FROZEN_2026-08-30",
    "target": "T1_FIRST_INQUIRY_EVENTUAL_SCHEDULED_VISIT_V1",
    "capacity_pct": 20,
    "fallback_k": 3,
}

assert post["phase"] == "FROZEN"
assert post["recovered_lead_quality"]["version"] == EXPECTED["lead_quality"]
assert post["recovered_lead_quality"]["target"] == EXPECTED["target"]
assert post["opportunity"]["version"] == EXPECTED["opportunity"]
assert post["capacity"]["selected_pct"] == EXPECTED["capacity_pct"]
assert post["inventory"]["fallback_max_k"] == EXPECTED["fallback_k"]
assert audit["status"] == "READY"
assert audit["gate"]["blocker_count"] == 0
assert llm_gate["status"] == "PASS"
assert score_cfg["formula"]["internal_0_1"] == "lead_quality_probability * inventory_actionability_gate"

print("PASS — post-recovery authorities are consistent.")

## 2. Business question and scoring moment

The system separates:

- **Lead Quality:** who is more likely to reach the target outcome?
- **Inventory Serviceability:** can the current/fallback inventory serve the lead?
- **Lead Opportunity:** where do progression propensity and actionability coincide?

The final scoring moment is **T1**, immediately after the first inquiry arrives and the selected Spot is known.

The frozen target asks:

> **Will this first inquiry eventually be recorded as `scheduled_visit`?**

Target maturity is 14 days. The target was frozen before model selection.

In [ ]:
t = snapshot["target"]
s = snapshot["split"]
print("Target:", t["version"])
print("Maturity days:", t["maturity_days"])
print("Target coverage:", t["coverage"])
print("DEVELOPMENT:", s["development_n"])
print("OOF validation rows:", s["oof_validation_n"])
print("Procedural holdout:", s["procedural_holdout_status"])

## 3. Lead Quality recovery

The pre-recovery state did not provide acceptable prioritization signal. Prompt 11.5 recovered signal **without changing target or temporal splits** and without adding Availability.

Final champion: small regularized Logistic Regression with:
1. `selected_spot_area_closeness`
2. `selected_spot_geographic_fit`
3. `selected_spot_attribute_completeness`

The signal is modest rather than high-separation. That limitation is part of the final decision.

In [ ]:
m = snapshot["lead_quality"]["recovery_metrics"]
for k in ["lift5","lift10","lift20","average_precision","base_rate_average_precision","roc_auc","brier"]:
    print(f"{k:28s}: {m[k]}")
print("Lift@10 > 1 folds:", m["lift10_gt1_folds"])
print("Lift@20 > 1 folds:", m["lift20_gt1_folds"])
print("Delta Lift@10 CI95:", m["delta_lift10_ci95"])

## 4. Capacity frontier

Capacity was selected only from **DEVELOPMENT temporal OOF**.

The top-5 slice remains below random. The final P80/top20 policy is not hidden tuning: it is the strongest passing 10/15/20 clean-room capacity while retaining the most positives.

In [ ]:
with (A / "opportunity_score" / "outputs" / "capacity_metrics.csv").open(newline="") as f:
    rows = list(csv.DictReader(f))

frontier = [
    r for r in rows
    if r["population"] == "DEVELOPMENT_OOF"
    and r["aggregation"] == "MACRO"
    and r["system"] == "OPPORTUNITY_ACTIONABILITY_GATE_V2"
    and r["objective"] == "LEAD_QUALITY"
]
assert [int(r["capacity_pct"]) for r in frontier] == [5,10,15,20]

print("capacity | lift  | recall | precision | status")
for r in frontier:
    print(f'{int(r["capacity_pct"]):>3}%     | {float(r["lift"]):.3f} | {float(r["recall"]):.3f}  | {float(r["precision"]):.3f}     | {r["selection_status"]}')

## 5. Inventory and fallback

Availability uses backward as-of state: the latest snapshot known at score time.

The assessment does not use future `days_until_available` semantics to create artificial historical precision.

Fallback is capped at **K=3**. `NO_RESULT` is preferable to indefinite relaxation.

In [ ]:
inv_cfg = json.loads((A / "inventory" / "frozen_inventory_config.json").read_text())
rev = inv_cfg["post_recovery_fallback_revision"]

assert inv_cfg["fallback"]["max_recommendations"] == 3
print("Fallback K:", inv_cfg["fallback"]["max_recommendations"])
print("Selection population:", rev["selection_population"])
print("Reason:", rev["reason"])
print("Availability join:", inv_cfg["availability"]["join"])

## 6. Opportunity Score and the central trade-off

Historical E020 established the useful concept of combining Quality and Inventory, but it is **not** the final score authority.

After recovery, Lead Quality already contains selected-Spot matching context. Multiplying continuous Inventory Serviceability again caused double counting.

At top15, the rejected raw product improved exact-serviceable joint concentration but pushed Lead Quality Lift below 1.

Final V2 therefore uses only an actionability gate:

`Opportunity Score = P_quality × inventory_actionability_gate`

It is an operational score, **not a jointly calibrated probability**.

In [ ]:
tr = snapshot["tradeoff"]
print("Top15 recovered Lead Quality Lift:", round(tr["lead_quality_recovered_lift"], 3))
print("Top15 raw product Lead Quality Lift:", round(tr["rejected_raw_product_lead_quality_lift"], 3))
print("Top15 raw product joint-exact Lift:", round(tr["rejected_raw_product_joint_exact_lift"], 3))
print()
print(snapshot["opportunity"]["formula"])
print(snapshot["opportunity"]["meaning"])

### Decision rule

Use **Lead Quality** when the objective is:

> maximize scheduled visits regardless of inventory.

Use **Opportunity Score** when the objective is:

> prioritize leads likely to progress **and** serviceable with current/fallback inventory.

## 7. LLM / AI

The assessment uses a real LLM where the data genuinely contains unstructured language: listing copy.

Canonical E017:
- `gpt-5-nano`
- 100 records
- Structured Outputs
- cost ≈ **USD 0.002579**
- 0/100 new rule candidates
- 0/100 residual actionable

The reusable patterns became deterministic Rules-first checks.

Final LLM role: **sampled Semantic Inventory / Catalog QA discovery**.

No LLM inference is required to reproduce Lead Quality or Opportunity Score. Human precision/recall is unavailable because no complete human-gold set exists.

In [ ]:
l = snapshot["llm"]
print("LLM role:", l["role"])
print("Canonical E017 cost USD:", l["canonical_e017"]["cost_usd"])
print("New rule candidates:", l["canonical_e017"]["new_rule_candidates"])
print("Residual actionable:", l["canonical_e017"]["residual_actionable"])
print("Main score requires live OpenAI:", l["main_score_requires_live_openai"])
print("Human precision/recall:", l["human_precision_recall"])

## 8. Final architecture boundaries

Closed upstream lines remain closed:

- Matching / clusters = **AUXILIARY**
- Semantic Rules = **INVENTORY / CATALOG QA**
- Response-time RF = **DIAGNOSTIC ONLY**

None is reintroduced as a central Lead Opportunity component.

In [ ]:
print(snapshot["final_architecture"])

## 9. Limitations

The final assessment deliberately preserves:

1. June procedural holdout is non-pristine / diagnostic-only.
2. Top-5 Lead Quality Lift remains below 1.
3. Recovery uncertainty is wide.
4. Historical Spot prices are unversioned; precise budget fit is blocked/unknown.
5. Real score ties require rank-based priority bands.
6. No causal or commercial-conversion claim is supported.
7. Opportunity Score is not a jointly calibrated probability.
8. LLM human precision/recall is unavailable.

In [ ]:
for i, item in enumerate(snapshot["limitations"], 1):
    print(f"{i}. {item}")

## 10. Final packaging gate

This last cell is intentionally strict. It checks the exact authorities that all final deliverables must use.

In [ ]:
assert snapshot["lead_quality"]["champion"] == EXPECTED["lead_quality"]
assert snapshot["opportunity"]["version"] == EXPECTED["opportunity"]
assert snapshot["capacity"]["policy"] == "P80 / top 20% within T1"
assert snapshot["inventory"]["fallback_k"] == 3
assert snapshot["llm"]["main_score_requires_live_openai"] is False
assert snapshot["final_architecture"]["semantic_rules"] == "INVENTORY_CATALOG_QA"

print("FINAL NOTEBOOK CONSISTENCY: PASS")
print("Champion:", EXPECTED["lead_quality"])
print("Opportunity:", EXPECTED["opportunity"])
print("Capacity: P80 / top 20%")
print("Fallback: K=3")